# Reference identification baseline

Reconstruct the reference search corpus from the study exports.

Method and inputs: [workflow guide](../../docs/workflow.md).

In [1]:
import sys
from pathlib import Path

for _parent in (Path.cwd(), *Path.cwd().parents):
    if (_parent / "notebooks" / "_shared.py").is_file():
        sys.path.insert(0, str(_parent / "notebooks"))
        break
else:
    raise FileNotFoundError("Start Jupyter inside the repository.")

from _shared import (
    project_root, data_directory, results_directory, load_environment,
    require_environment, cache_directory, cache_fingerprint,
    load_cached_index, save_cached_index, metadata_directory, pdf_directory,
    reference_datasets, review_records, case_study_filters, align_full_text_results,
)

load_environment()

import json
import pandas as pd

folder_path = data_directory()
metadata_path = metadata_directory("reference")
result_path = results_directory("reference")

ieee_file = metadata_path + 'ieee.csv'
scopus_file = metadata_path + 'scopus.csv'
wos_file = metadata_path + 'wos.xls'

## IEEE Xplore

In [2]:
df_ieee = pd.read_csv(ieee_file)

df_ieee = df_ieee[pd.to_datetime(df_ieee['Date Added To Xplore'], errors='coerce') < pd.Timestamp('2024-07-17')]

df_ieee = df_ieee[['DOI', 'Document Title', 'Authors', 'Publication Year', 'Abstract', 'Article Citation Count']]

df_ieee = df_ieee.rename(columns={
    'Document Title': 'Title',
    'Publication Year': 'Year',
    'Article Citation Count': 'Citations'
})

df_ieee = df_ieee.dropna(subset=['DOI'])
# df_ieee = df_ieee.dropna(subset=['Citations'])
df_ieee

,DOI,Title,Authors,Year,Abstract,Citations
1,10.1109/VCIP59821.2023.10402700,End-to-end Action Quality Assessment with Acti...,H. Fang; W. Zhou; H. Li,2023,Action Quality Assessment (AQA) plays a crucia...,5.0
2,10.1109/ICIP42928.2021.9506257,Action Quality Assessment With Ignoring Scene ...,T. Nagai; S. Takeda; M. Matsumura; S. Shimizu;...,2021,We propose an action quality assessment (AQA) ...,6.0
3,10.1109/ICAACE61206.2024.10548995,Action Quality Assessment with Multi-scale Tem...,W. Wang; H. Wang; Y. Hao; Q. Wang,2024,Action quality assessment is a more challengin...,NaN
5,10.1109/ICASSP48485.2024.10447069,Multi-Stage Contrastive Regression for Action ...,Q. An; M. Qi; H. Ma,2024,"In recent years, there has been growing intere...",7.0
6,10.1109/ITME53901.2021.00048,Skeleton Based Action Quality Assessment of Fi...,H. -Y. Li; Q. Lei; H. -B. Zhang; J. -X. Du,2021,Action quality assessment(AQA) aims at achievi...,7.0
8,10.1109/INSAI54028.2021.00029,A Survey of Video-based Action Quality Assessment,S. Wang; D. Yang; P. Zhai; Q. Yu; T. Suo; Z. S...,2021,Human action recognition and analysis have gre...,17.0
9,10.1109/TCSVT.2022.3143549,Semi-Supervised Action Quality Assessment With...,S. -J. Zhang; J. -H. Pan; J. Gao; W. -S. Zheng,2022,Action Quality Assessment aims to evaluate how...,29.0
10,10.1109/ICIP.2018.8451364,S3D: Stacking Segmental P3D for Action Quality...,X. Xiang; Y. Tian; A. Reiter; G. D. Hager; T. ...,2018,Action quality assessment is crucial in areas ...,45.0
11,10.1109/TIP.2023.3331212,Fine-Grained Spatio-Temporal Parsing Network f...,K. Gedamu; Y. Ji; Y. Yang; J. Shao; H. T. Shen,2023,Action Quality Assessment (AQA) plays an impor...,16.0
12,10.1109/ICASSP49357.2023.10095031,Attention-Guided Deep Learning Framework For M...,A. Kanade; M. Sharma; M. Muniyandi,2023,Physical rehabilitation programs frequently be...,5.0


## Scopus

In [3]:
df_scopus = pd.read_csv(scopus_file)

df_scopus = df_scopus[['DOI', 'Title', 'Authors', 'Year', 'Abstract', 'Cited by']]

df_scopus = df_scopus.rename(columns={
    'Cited by': 'Citations'
})

df_scopus = df_scopus.dropna(subset=['DOI'])
# df_scopus = df_scopus.loc[df_scopus['Citations'] > 0]
df_scopus

,DOI,Title,Authors,Year,Abstract,Citations
0,10.3390/s24237455,A Spatial-Temporal Multi-Feature Network (STMF...,"Tian, Y.; Lin, S.; Xu, H.; Chen, G.",2024,"Globally, monitoring productivity, occupationa...",0
1,10.1007/s11263-024-02146-z,Procedure-Aware Action Quality Assessment: Dat...,"Xu, J.; Rao, Y.; Zhou, J.; Lu, J.",2024,"In this paper, we investigate the problem of p...",5
2,10.1145/3664647.3681084,2M-AF: A Strong Multi-Modality Framework for H...,"Ding, Y.; Zhang, S.; Liu, L.; Zhang, J.; Chen,...",2024,Human Action Quality Assessment (AQA) is a pro...,1
3,10.1049/cvi2.12287,SkatingVerse: A large-scale benchmark for comp...,"Gan, Z.; Jin, L.; Cheng, Y.; Cheng, Y.; Teng, ...",2024,Human action understanding (HAU) is a broad to...,4
4,10.1007/s13735-024-00338-4,Human action recognition using an optical flow...,"Giveki, D.",2024,Recognizing various human actions in videos is...,10
...,...,...,...,...,...,...
136,10.1109/JBHI.2014.2375206,Component-level tuning of kinematic features f...,"Venkataraman, V.; Turaga, P.; Baran, M.; Lehre...",2016,"In this paper, we propose a general framework ...",10
137,10.1109/EMBC.2014.6944292,Decision support for stroke rehabilitation the...,"Venkataraman, V.; Turaga, P.; Lehrer, N.; Bara...",2014,This paper proposes a computational framework ...,13
138,10.1007/s11042-012-1097-x,An audio-visual approach to web video categori...,"Ionescu, B.E.; Seyerlehner, K.; Mironicǎ, I.; ...",2014,"In this paper, we discuss and audio-visual app...",10
139,10.1145/2505323.2505330,Graph-based analysis of physical exercise actions,"Çeliktutan, O.; Akgül, C.B.; Wolf, C.; Sankur, B.",2013,"In this paper, we develop a graph-based method...",30


In [4]:
dois_to_remove = [
    "10.3390/s24237455",
    "10.1145/3664647.3681084",
    "10.1109/ICMRA62519.2024.10809303",
    "10.1007/978-981-97-7232-2_31",
    "10.1080/21680566.2024.2380913",
    "10.1109/ICLT63507.2024.11038644"
]

df_scopus = df_scopus[~df_scopus["DOI"].isin(dois_to_remove)].reset_index(drop=True)
df_scopus

,DOI,Title,Authors,Year,Abstract,Citations
0,10.1007/s11263-024-02146-z,Procedure-Aware Action Quality Assessment: Dat...,"Xu, J.; Rao, Y.; Zhou, J.; Lu, J.",2024,"In this paper, we investigate the problem of p...",5
1,10.1049/cvi2.12287,SkatingVerse: A large-scale benchmark for comp...,"Gan, Z.; Jin, L.; Cheng, Y.; Cheng, Y.; Teng, ...",2024,Human action understanding (HAU) is a broad to...,4
2,10.1007/s13735-024-00338-4,Human action recognition using an optical flow...,"Giveki, D.",2024,Recognizing various human actions in videos is...,10
3,10.1016/j.compbiomed.2024.108875,A novel biomechanical index for quality assess...,"Abedi, M.; Behzadipour, S.",2024,Background: While motor recovery is preferred ...,1
4,10.1016/j.compbiomed.2024.108382,LightPRA: A Lightweight Temporal Convolutional...,"Sardari, S.; Sharifzadeh, S.; Daneshkhah, A.; ...",2024,Research evidence shows that physical rehabili...,7
...,...,...,...,...,...,...
126,10.1109/JBHI.2014.2375206,Component-level tuning of kinematic features f...,"Venkataraman, V.; Turaga, P.; Baran, M.; Lehre...",2016,"In this paper, we propose a general framework ...",10
127,10.1109/EMBC.2014.6944292,Decision support for stroke rehabilitation the...,"Venkataraman, V.; Turaga, P.; Lehrer, N.; Bara...",2014,This paper proposes a computational framework ...,13
128,10.1007/s11042-012-1097-x,An audio-visual approach to web video categori...,"Ionescu, B.E.; Seyerlehner, K.; Mironicǎ, I.; ...",2014,"In this paper, we discuss and audio-visual app...",10
129,10.1145/2505323.2505330,Graph-based analysis of physical exercise actions,"Çeliktutan, O.; Akgül, C.B.; Wolf, C.; Sankur, B.",2013,"In this paper, we develop a graph-based method...",30


## Web of Science

In [5]:
df_wos = pd.read_excel(wos_file)

df_wos = df_wos[pd.to_datetime(df_wos['Indexed Date'], errors='coerce') < pd.Timestamp('2024-07-17')]

df_wos = df_wos[['DOI', 'Article Title', 'Authors', 'Publication Year', 'Abstract', 'Times Cited, All Databases']]
df_wos = df_wos.rename(columns={
    'Article Title': 'Title',
    'Publication Year': 'Year',
    'Times Cited, All Databases': 'Citations'
})

df_wos = df_wos.dropna(subset=['DOI', 'Abstract'])
df_wos = df_wos.loc[df_wos['Year'] != 2025]
# df_wos = df_wos.loc[df_wos['Citations'] > 0]
df_wos

,DOI,Title,Authors,Year,Abstract,Citations
0,10.1109/TCSVT.2022.3143549,Semi-Supervised Action Quality Assessment With...,"Zhang, Shao-Jie; Pan, Jia-Hui; Gao, Jibin; Zhe...",2022,Action Quality Assessment aims to evaluate how...,26
1,10.3390/s19194129,A Survey of Vision-Based Human Action Evaluati...,"Lei, Qing; Du, Ji-Xiang; Zhang, Hong-Bo; Ye, S...",2019,The fields of human activity analysis have rec...,81
2,10.3390/s23229282,Action Quality Assessment Model Using Speciali...,"Hirosawa, Seiji; Kato, Takaaki; Yamashita, Tak...",2023,Action quality assessment (AQA) tasks in compu...,6
4,10.1016/j.patrec.2022.04.015,Learning time-aware features for action qualit...,"Zhang, Yu; Xiong, Wei; Mi, Siya",2022,Action quality assessment (AQA) is a task to a...,8
5,10.1007/s11760-021-01890-w,Temporal attention learning for action quality...,"Lei, Qing; Zhang, Hongbo; Du, Jixiang",2021,This paper proposes an end-to-end temporal att...,15
6,10.1016/j.knosys.2021.107388,Learning and fusing multiple hidden substages ...,"Dong, Li-Jia; Zhang, Hong-Bo; Shi, Qinghongya;...",2021,Many of the existing methods for action qualit...,23
7,10.1007/s10489-022-03984-5,Label-reconstruction-based pseudo-subscore lea...,"Zhang, Hong-Bo; Dong, Li-Jia; Lei, Qing; Yang,...",2023,Most existing action quality assessment (AQA) ...,5
10,10.1109/TIP.2023.3331212,Fine-Grained Spatio-Temporal Parsing Network f...,"Gedamu, Kumie; Ji, Yanli; Yang, Yang; Shao, Ji...",2023,Action Quality Assessment (AQA) plays an impor...,14
11,10.1016/j.compbiomed.2020.104021,Automated classification of movement quality u...,"Dajime, Peter Fermin; Smith, Heather; Zhang, Y...",2020,Practitioners commonly perform movement qualit...,13
12,10.1007/s40747-022-00892-6,Gaussian guided frame sequence encoder network...,"Li, Ming-Zhe; Zhang, Hong-Bo; Dong, Li-Jia; Le...",2023,Can a computer evaluate an athlete's performan...,6


## Final Dataset

In [6]:
df_all = pd.concat([df_ieee, df_scopus, df_wos], ignore_index=True)
df_all = df_all.drop_duplicates(subset=['DOI'])
df_all = df_all.drop(columns=['Citations'])
df_all = df_all.reset_index(drop=True)
df_all

,DOI,Title,Authors,Year,Abstract
0,10.1109/VCIP59821.2023.10402700,End-to-end Action Quality Assessment with Acti...,H. Fang; W. Zhou; H. Li,2023,Action Quality Assessment (AQA) plays a crucia...
1,10.1109/ICIP42928.2021.9506257,Action Quality Assessment With Ignoring Scene ...,T. Nagai; S. Takeda; M. Matsumura; S. Shimizu;...,2021,We propose an action quality assessment (AQA) ...
2,10.1109/ICAACE61206.2024.10548995,Action Quality Assessment with Multi-scale Tem...,W. Wang; H. Wang; Y. Hao; Q. Wang,2024,Action quality assessment is a more challengin...
3,10.1109/ICASSP48485.2024.10447069,Multi-Stage Contrastive Regression for Action ...,Q. An; M. Qi; H. Ma,2024,"In recent years, there has been growing intere..."
4,10.1109/ITME53901.2021.00048,Skeleton Based Action Quality Assessment of Fi...,H. -Y. Li; Q. Lei; H. -B. Zhang; J. -X. Du,2021,Action quality assessment(AQA) aims at achievi...
...,...,...,...,...,...
147,10.1109/TASE.2016.2628091,Online Reconfiguration of Active Distribution ...,"Koutsoukis, Nikolaos C.; Siagkas, Dimitris O.;...",2017,This paper proposes the online reconfiguration...
148,10.1177/14604582231214589,A comparative study of the 2D-and 3D-based ske...,"Lincke, Alisa; Fagerstrom, Cecilia; Ekstedt, M...",2023,Background: Maintaining physical activity (PA)...
149,10.1016/j.jclepro.2023.139357,The impact of digital inclusive finance on the...,"Wei, Danqi; Ahmad, Fayyaz; Abid, Nabila; Khan,...",2023,The development of renewable energy is the key...
150,10.1038/s41558-018-0343-2,An assessment of climate action by high-carbon...,"Dietz, Simon; Fruitiere, Charles; Garcia-Manas...",2018,Corporations are an important source of GHG em...


In [7]:
df_all.to_csv(result_path + "identified_records.csv", index=False)

In [8]:
# Step 1: Load the JSON file
study_data = reference_datasets()

# Step 2: Extract the list of DOIs (keys)
study_dois = list(study_data.keys())
print(f"Total DOIs in JSON: {len(study_dois)}")

# Step 3: Ensure df_all DOI column is string type
df_all_dois = df_all['DOI'].astype(str)

# Step 4: Separate DOIs into present and not present
dois_present = [doi for doi in study_dois if doi in df_all_dois.values]
dois_not_present = [doi for doi in study_dois if doi not in df_all_dois.values]

# Step 5: Print results
print(f"Number of JSON DOIs present in df_all: {len(dois_present)}")
print("DOIs present:", dois_present)
print(f"\nNumber of JSON DOIs NOT present in df_all: {len(dois_not_present)}")
print("DOIs not present:", dois_not_present)

Total DOIs in JSON: 94
Number of JSON DOIs present in df_all: 92
DOIs present: ['10.1145/2505323.2505330', '10.1109/CVPRW.2013.82', '10.1109/EMBC.2014.6944292', '10.1109/ICCT.2015.7399879', '10.1109/JBHI.2014.2375206', '10.1109/TPAMI.2016.2533388', '10.1007/978-3-030-00767-6_12', '10.1109/ICIP.2018.8451364', '10.1109/CVPR.2019.00039', '10.1007/978-3-030-20876-9_10', '10.1109/ICCV.2019.00643', '10.1109/WACV.2019.00161', '10.1016/j.compbiomed.2020.104021', '10.1109/TransAI49837.2020.00030', '10.1145/3341105.3374092', '10.1109/CVPR42600.2020.00986', '10.1145/3394171.3413560', '10.3390/electronics9040568', '10.1007/978-3-030-58577-8_14', '10.1007/978-3-030-60639-8_1', '10.1109/WACV48630.2021.00044', '10.1109/ICCV48922.2021.00782', '10.1109/ICIP42928.2021.9506257', '10.1109/MMSP53017.2021.9733638', '10.1109/ITME53901.2021.00048', '10.1016/j.knosys.2021.107388', '10.1145/3474085.3475438', '10.1145/3453892.3461624', '10.1007/s11760-021-01890-w', '10.1145/3462244.3479891', '10.1016/j.patcog.20